# Repair Results Analysis

This notebook loads repair results from SQLite databases and generates summary statistics and visualizations.


In [ ]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
%matplotlib inline

In [ ]:
database_version = "6"

In [ ]:
target_db = f'targets{database_version}.db'

In [ ]:
sources_dbs = [
            "repair_results_gemini_gemini-2.5-flash_targets6.db",
            "repair_results_ollama_llama3:70b_targets6.db",
            "repair_results_openai_gpt-4.1-2025-04-14_targets6.db"
            ]

In [ ]:
import sqlite3
conn = sqlite3.connect(target_db)
# Note: cases.id in targets DB corresponds to source puzzle.id
df_cases = pd.read_sql(
    "SELECT id AS puzzle_id, num_nonterminals, nonterminal_prob, loop_prob, mutation_depth, original_parser, corrupted_grammar FROM cases",
    conn
)

# Compute derived columns for analysis
df_cases['parser_size'] = df_cases['original_parser'].str.len()
df_cases['corrupted_symbol_count'] = df_cases['corrupted_grammar'].str.split().str.len()

In [ ]:
import matplotlib.pyplot as plt

df_cases['num_nonterminals'] = pd.to_numeric(df_cases['num_nonterminals'], errors='coerce')
df_cases = df_cases.dropna(subset=['num_nonterminals'])
df_cases['num_nonterminals'] = df_cases['num_nonterminals'].astype(int)

all_num_nts = sorted(df_cases['num_nonterminals'].unique())
models = []

records = []
for db_path in sources_dbs:
    model = os.path.basename(db_path).removeprefix('repair_results_').removesuffix('_targets.db').split('_')[1]
    models.append(model)

    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql('SELECT case_id, fix FROM repair_results', conn)
    conn.close()

    df_ok = df_res[df_res['fix'] == 1].merge(df_cases[['puzzle_id', 'num_nonterminals']], left_on='case_id', right_on='puzzle_id')
    counts = df_ok['num_nonterminals'].value_counts().to_dict()

    for num_nt in all_num_nts:
        cnt = counts.get(num_nt, 0)  
        records.append({'model': model, 'num_nonterminals': num_nt, 'accuracy': cnt})

df_nt_fixes = pd.DataFrame(records)

plt.figure(figsize=(10, 6))
for model in sorted(set(models)):
    df_model = df_nt_fixes[df_nt_fixes['model'] == model].sort_values('num_nonterminals')
    plt.plot(df_model['num_nonterminals'], df_model['accuracy'], marker='o', label=model)

plt.xlabel('Number of Nonterminals')
plt.ylabel('Accuracy')
plt.title('Accuracy by Number of Nonterminals Across Models')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.xticks(all_num_nts, rotation=45)
# plt.ylim(0, 100)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of corrupted_symbol_count for successful fixes per model
records = []
for db_path in sources_dbs:
    model = os.path.basename(db_path).removeprefix('repair_results_').removesuffix('_targets.db')
    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql(
        'SELECT case_id, fix FROM repair_results',
        conn
    )
    conn.close()
    df_ok = df_res[df_res['fix'] == 1].merge(
        df_cases[['puzzle_id', 'corrupted_symbol_count']], left_on='case_id', right_on='puzzle_id'
    )
    for cs in df_ok['corrupted_symbol_count']:
        records.append({'model': model, 'corrupted_symbol_count': cs})

df_okcases = pd.DataFrame(records)

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_okcases, x='model', y='corrupted_symbol_count', palette='tab10')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Model')
plt.ylabel('Symbol Count')
plt.title('Distribution of Symbol(Nonterminal + Terminal) Count for Successful Fixes per Model')
plt.tight_layout()


In [ ]:
# Bar plot of accuracy by number of nonterminals across all models
# Ensure num_nonterminals is integer for correct sorting and plotting
df_cases['num_nonterminals'] = pd.to_numeric(df_cases['num_nonterminals'], errors='coerce')
df_cases = df_cases.dropna(subset=['num_nonterminals'])
df_cases['num_nonterminals'] = df_cases['num_nonterminals'].astype(int)

records = []
for db_path in sources_dbs:
    model = os.path.basename(db_path).removeprefix('repair_results_').removesuffix('_targets.db').split('_')[1]
    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql('SELECT case_id, fix FROM repair_results', conn)
    conn.close()
    df_ok = df_res[df_res['fix'] == 1].merge(df_cases[['puzzle_id','num_nonterminals']], left_on='case_id', right_on='puzzle_id')
    counts = df_ok['num_nonterminals'].value_counts().sort_index()
    for num_nt, cnt in counts.items():
        records.append({'model': model, 'num_nonterminals': num_nt, 'accuracy': cnt})

df_nt_fixes = pd.DataFrame(records)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_nt_fixes, x='num_nonterminals', y='accuracy', hue='model')
plt.xlabel('Number of Nonterminals')
plt.ylabel('Accuracy')
plt.title('Accuracy by Number of Nonterminals Across Models')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()


In [ ]:
import matplotlib.pyplot as plt

df_cases['num_nonterminals'] = pd.to_numeric(df_cases['num_nonterminals'], errors='coerce')
df_cases = df_cases.dropna(subset=['num_nonterminals'])
df_cases['num_nonterminals'] = df_cases['num_nonterminals'].astype(int)

all_num_nts = sorted(df_cases['num_nonterminals'].unique())
models = df_nt_fixes['model'].unique()

for model in models:
    df_model_raw = df_nt_fixes[df_nt_fixes['model'] == model]
    df_model = pd.DataFrame({'num_nonterminals': all_num_nts})
    df_model = df_model.merge(df_model_raw[['num_nonterminals', 'accuracy']], on='num_nonterminals', how='left')
    df_model['accuracy'] = df_model['accuracy'].fillna(0).astype(int)
    
    plt.figure(figsize=(10, 6))
    plt.plot(df_model['num_nonterminals'], df_model['accuracy'], marker='o', linestyle='-')


    plt.title(f'Accuracy for {model} by Number of Nonterminals')
    plt.xlabel('Number of Nonterminals')
    plt.ylabel('Accuracy')
    plt.xticks(all_num_nts, rotation=45)
    # plt.ylim(0, 100)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Bar plot of average mutation_depth vs num_nonterminals
avg_mut = df_cases.groupby('num_nonterminals')['mutation_depth'].mean().reset_index()
plt.figure(figsize=(8, 6))
sns.barplot(data=avg_mut, x='num_nonterminals', y='mutation_depth', palette='tab10')
plt.xlabel('Number of Nonterminals')
plt.ylabel('Average Mutation Depth')
plt.title('Average Mutation Depth vs Number of Nonterminals')
plt.tight_layout()


In [ ]:
# Combined heatmap of fix-rate across all models by nonterminal_prob and loop_prob
records = []
for db_path in sources_dbs:
    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql(
        'SELECT nonterminal_prob, loop_prob, fix FROM repair_results', conn
    )
    conn.close()
    df_res['model'] = os.path.basename(db_path)
    records.append(df_res)

df_all = pd.concat(records, ignore_index=True)

totals = df_all.groupby(['nonterminal_prob','loop_prob']).size().rename('total')
successes= df_all[df_all['fix'] == 1].groupby(['nonterminal_prob','loop_prob']).size().rename('success')
df_rate = pd.concat([totals, successes], axis=1).fillna(0)
df_rate['rate'] = df_rate['success'] / df_rate['total']
pivot = df_rate['rate'].unstack(level='loop_prob')

plt.figure(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Combined Fix Rate Across All Models')
plt.xlabel('loop_prob')
plt.ylabel('nonterminal_prob')
plt.tight_layout()


In [ ]:
# Scatter plot of parser_size vs corrupted_symbol_count, highlighting fixed cases per model
for db_path in sources_dbs:
    model = os.path.basename(db_path).removeprefix('repair_results_').removesuffix('_targets.db')
    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql(
        'SELECT case_id, fix FROM repair_results', conn
    )
    conn.close()
    fixed = df_res[df_res['fix'] == 1]['case_id']
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=df_cases, x='corrupted_symbol_count', y='parser_size', alpha=0.3, label='All cases')
    sns.scatterplot(data=df_cases[df_cases['puzzle_id'].isin(fixed)], 
                    x='corrupted_symbol_count', y='parser_size', color='red', label='Fixed cases')
    plt.title(f'Parser Size vs Symbol Count (Fixed) for {model}')
    plt.xlabel('Symbol Count')
    plt.ylabel('Parser Size')
    plt.legend()
    plt.tight_layout()


In [ ]:
# Combined scatter plot of fixed cases across all models
records = []
for db_path in sources_dbs:
    model = os.path.basename(db_path).removeprefix('repair_results_').removesuffix('_targets.db')
    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql('SELECT case_id, puzzle_id, fix, total_tokens FROM repair_results', conn)
    # Merge using cases.id (puzzle_id) mapping to source puzzle.id
    df_tmp = df_res[df_res['fix'] == 1].merge(df_cases, left_on='case_id', right_on='puzzle_id')
    df_tmp['model'] = model
    records.append(df_tmp)
    conn.close()

df_fixed_all = pd.concat(records, ignore_index=True)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_cases, x='corrupted_symbol_count', y='parser_size', color='gray', alpha=0.3, label='All cases')
sns.scatterplot(data=df_fixed_all, x='corrupted_symbol_count', y='parser_size', hue='puzzle_id_x')
plt.xlabel('Symbol Count')
plt.ylabel('Parser Size')
plt.title('Fixed Cases Across All Models')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()


In [ ]:
# Scatter plot of token consumption vs number of nonterminals across all fixed cases
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_fixed_all, x='num_nonterminals', y='total_tokens', hue='model', alpha=0.7)
plt.xlabel('Number of Nonterminals')
plt.ylabel('Total Tokens')
plt.title('Token Consumption vs Number of Nonterminals (All Models)')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Individual model plots with linear fit
for model in df_fixed_all['model'].unique():
    df_mod = df_fixed_all[df_fixed_all['model'] == model]
    plt.figure(figsize=(8, 6))

    sns.scatterplot(
        data=df_mod,
        x='num_nonterminals',
        y='total_tokens',
        color='C0',
        alpha=0.7,
        label='data'
    )
    sns.regplot(
        data=df_mod,
        x='num_nonterminals',
        y='total_tokens',
        scatter=False,      # 不画散点
        line_kws={'color': 'red', 'lw': 2},
        label='linear fit',
        order=2
    )

    plt.xlabel('Number of Nonterminals')
    plt.ylabel('Total Tokens')
    plt.title(f'Token Consumption vs Number of Nonterminals ({model})')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Scatter plot of token consumption vs mutation depth across all fixed casesplt.figure(figsize=(8, 6))sns.scatterplot(data=df_fixed_all, x='mutation_depth', y='total_tokens', hue='model', alpha=0.7)plt.xlabel('Mutation Depth')plt.ylabel('Total Tokens')plt.title('Token Consumption vs Mutation Depth (All Models)')plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')plt.tight_layout()# Individual model plots with quadratic fitfor model in df_fixed_all['model'].unique():    df_mod = df_fixed_all[df_fixed_all['model'] == model]    plt.figure(figsize=(8, 6))    sns.scatterplot(data=df_mod, x='mutation_depth', y='total_tokens', color='C0', alpha=0.7, label='data')    sns.regplot(data=df_mod, x='mutation_depth', y='total_tokens', scatter=False, line_kws={'color': 'red', 'lw': 2}, label='quadratic fit', order=2)    plt.xlabel('Mutation Depth')    plt.ylabel('Total Tokens')    plt.title(f'Token Consumption vs Mutation Depth ({model})')    plt.legend()    plt.tight_layout()    plt.show()

In [ ]:
# Table of fixed cases counts by mutation_depth for each model
records = []
for db_path in sources_dbs:
    model = os.path.basename(db_path).removeprefix('repair_results_').removesuffix('_targets.db')
    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql('SELECT case_id, fix FROM repair_results', conn)
    conn.close()
    df_ok = df_res[df_res['fix'] == 1].merge(df_cases[['puzzle_id','mutation_depth']], left_on='case_id', right_on='puzzle_id')
    counts = df_ok['mutation_depth'].value_counts().sort_index()
    for depth, cnt in counts.items():
        records.append({'model': model, 'mutation_depth': depth, 'accuracy': cnt})

df_mut_counts = pd.DataFrame(records)
pivot_mut = df_mut_counts.pivot(index='mutation_depth', columns='model', values='accuracy').fillna(0).astype(int)
print(pivot_mut)


In [ ]:
records = []
for db_path in sources_dbs:
    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql(
        'SELECT nonterminal_prob, loop_prob, fix FROM repair_results', conn
    )
    conn.close()
    df_res['model'] = os.path.basename(db_path)
    records.append(df_res)

df_all = pd.concat(records, ignore_index=True)

# compute total & successful counts per (nonterminal_prob, loop_prob)
totals    = df_all.groupby(['nonterminal_prob','loop_prob']).size().rename('total')
successes = df_all[df_all['fix'] == 1] \
                    .groupby(['nonterminal_prob','loop_prob']).size().rename('success')
df_rate   = pd.concat([totals, successes],
axis=1).fillna(0)
df_rate['rate'] = df_rate['success'] / df_rate['total']

# pivot into heatmap-friendly form
pivot = df_rate['rate'].unstack(level='loop_prob')

# plot
plt.figure(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Combined Fix Rate Across All Models')
plt.xlabel('loop_prob')
plt.ylabel('nonterminal_prob')
plt.tight_layout()

In [ ]:
for db_path in sources_dbs:
        model = os.path.basename(db_path).removeprefix('repair_results_') \
                             .removesuffix('_targets.db')
        conn = sqlite3.connect(db_path)
        df_res = pd.read_sql(
            'SELECT nonterminal_prob, loop_prob, fix FROM repair_results',
            conn
        )
        conn.close()

        # Compute success rate per (nonterminal_prob, loop_prob)
        totals   = df_res.groupby(['nonterminal_prob','loop_prob']).size().rename('total')
        successes= df_res[df_res['fix'] == 1] \
                        .groupby(['nonterminal_prob','loop_prob']).size().rename('success')
        df_rate  = pd.concat([totals, successes], axis=1).fillna(0)
        df_rate['rate'] = df_rate['success'] / df_rate['total']

        # Pivot for heatmap
        pivot = df_rate['rate'].unstack(level='loop_prob')

        plt.figure(figsize=(6, 5))
        sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu')
        plt.title(f'Fix Rate for {model}')
        plt.xlabel('loop_prob')
        plt.ylabel('nonterminal_prob')
        plt.tight_layout()

# Combined heatmap of fix-rate across all models by nonterminal_prob and loop_prob
records = []
for db_path in sources_dbs:
    conn = sqlite3.connect(db_path)
    df_res = pd.read_sql(
        'SELECT nonterminal_prob, loop_prob, fix FROM repair_results',
        conn
    )
    conn.close()
    df_res['model'] = os.path.basename(db_path)
    records.append(df_res)

df_all = pd.concat(records, ignore_index=True)

# Compute aggregated success rate
totals   = df_all.groupby(['nonterminal_prob','loop_prob']).size().rename('total')
successes= df_all[df_all['fix'] == 1] \
                .groupby(['nonterminal_prob','loop_prob']).size().rename('success')
df_rate  = pd.concat([totals, successes], axis=1).fillna(0)
df_rate['rate'] = df_rate['success'] / df_rate['total']

# Pivot for combined heatmap
pivot = df_rate['rate'].unstack(level='loop_prob')

plt.figure(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Combined Fix Rate Across All Models')
plt.xlabel('loop_prob')
plt.ylabel('nonterminal_prob')
plt.tight_layout()


In [ ]:
import matplotlib.pyplot as plt

# Step 1: 统计每个模型成功修复的 puzzle（按 mutation_depth）
records = []
for db_path in sources_dbs:
    model = os.path.basename(db_path).removeprefix('repair_results_') \
                                     .removesuffix('_targets.db')
    conn = sqlite3.connect(db_path)
    try:
        df_res = pd.read_sql("SELECT case_id, fix FROM repair_results", conn)
    except pd.io.sql.DatabaseError:
        conn.close()
        continue
    conn.close()

    # Merge on puzzle_id (cases.id) and case_id
    df_merged = df_res[df_res['fix'] == 1].merge(
        df_cases[['puzzle_id', 'mutation_depth']],
        left_on='case_id', right_on='puzzle_id'
    )

    # 按 puzzle_id 去重，避免重复计入多个 case
    df_unique = df_merged.drop_duplicates(subset='puzzle_id')
    counts = df_unique['mutation_depth'].value_counts().sort_index()
    for depth, cnt in counts.items():
        records.append({'model': model, 'mutation_depth': depth, 'fix_count': cnt})

df_counts = pd.DataFrame(records)

# Step 2: 计算 total_cases（按 puzzle_id 去重）
df_unique_puzzles = df_cases.drop_duplicates(subset='puzzle_id')
total_cases = df_unique_puzzles['mutation_depth'].value_counts().sort_index().to_dict()
all_depths = sorted(total_cases.keys())
all_models = df_counts['model'].unique()

# Step 3: 构造完整的 fix_rate 表格
full_records = []
for model in all_models:
    model_df = df_counts[df_counts['model'] == model].set_index('mutation_depth')['fix_count'].to_dict()
    for depth in all_depths:
        fix_count = model_df.get(depth, 0)
        total = total_cases.get(depth, 0)
        rate = fix_count / total if total > 0 else 0
        full_records.append({'model': model, 'mutation_depth': depth, 'fix_rate': rate})

df_rates = pd.DataFrame(full_records)

# Step 4: 绘图（每个模型折线）
plt.figure(figsize=(10, 6))
for model in df_rates['model'].unique():
    df_model = df_rates[df_rates['model'] == model].sort_values('mutation_depth')
    plt.plot(df_model['mutation_depth'], df_model['fix_rate'], marker='o', label=model)

plt.xlabel('Mutation Depth')
plt.ylabel('Fix Rate')
plt.title('Fix Rate by Mutation Depth (Unique Puzzle ID)')
plt.xticks(all_depths)
plt.ylim(0, 1)
plt.grid(True)
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Step 5: 绘图（所有模型的平均宽松准确率）
df_avg = df_rates.groupby('mutation_depth')['fix_rate'].mean().reset_index()

plt.figure(figsize=(8, 5))
plt.plot(df_avg['mutation_depth'], df_avg['fix_rate'], marker='o', linestyle='-')
plt.xlabel('Mutation Depth')
plt.ylabel('Average Fix Rate')
plt.title('Average Fix Rate by Mutation Depth (Unique Puzzle ID)')
plt.xticks(all_depths)
plt.ylim(0, 1)
plt.grid(True)
plt.tight_layout()
plt.show()

Pass@K

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

db_paths = [
 "repair_results_gemini_gemini-2.5-flash_targets6.db",
"repair_results_ollama_llama3:70b_targets6.db",
"repair_results_openai_gpt-4.1-2025-04-14_targets6.db"
]
ks = list(range(1, 6))
records = []

for db_path in db_paths:
    model = os.path.splitext(os.path.basename(db_path))[0]
    conn = sqlite3.connect(db_path)

    # Skip if there's no repair_results table
    existing = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='repair_results'",
        conn
    )
    if existing.empty:
        print(f"{db_path} has no repair_results table, skipping…")
        conn.close()
        continue

    # Check if 'sample' column exists
    info = pd.read_sql("PRAGMA table_info(repair_results)", conn)
    has_sample = 'sample' in info['name'].tolist()

    # Read required columns
    cols = "puzzle_id, fix" + (", sample" if has_sample else "")
    df = pd.read_sql(f"SELECT {cols} FROM repair_results", conn)
    conn.close()

    # If no sample column, assign sample numbers by puzzle_id occurrence order
    if not has_sample:
        df['sample'] = df.groupby('puzzle_id').cumcount() + 1

    total = df['puzzle_id'].nunique()
    for k in ks:
        df_k = df[df['sample'] <= k]
        # For each puzzle_id, if any of the first k attempts fixed it, count as passed
        passed = df_k.groupby('puzzle_id')['fix'].max().sum()
        rate = passed / total
        records.append({"model": model, "k": k, "rate": rate})

df_pass_k = pd.DataFrame(records)

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_pass_k, x='k', y='rate', hue='model', marker='o')
plt.xticks(ks, rotation=45)
plt.xlabel('k (max attempts)')
plt.ylabel('Pass@k Rate')
plt.title('Pass@k Rate per Model (from targets4 DB, sample by puzzle_id order)')
plt.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()